# Sprint E7 walkthrough: the alpha lab and backtest hygiene

In [1]:
# the repository root is importable so the package and the dashboard
# module can be imported without installing the wheel
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "efb").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
ALPHA = DATA / "alpha"

In [2]:
# Cell 1 rule: the data hash in the results file must be the hash of
# the artifacts the criteria are read from, recomputed now, not copied
from efb import evaluate

stored = json.loads(
    (ROOT / "sprints" / "E7" / "RESULTS.json").read_text()
)
gate = json.loads(
    (ROOT / "sprints" / "E7" / "RG_SIGNAL.json").read_text()
)
assert evaluate.e7_data_hash(DATA) == stored["data_hash"], "artifact hash drift"
print("data_hash", stored["data_hash"])
print("gate verdicts:",
      {name: block["verdict"] for name, block in gate.items()})

data_hash d530aad43f5d12e7e16433596c6624464c4779013a7b2f5761e6db8f2c7928b7
gate verdicts: {'idio_momentum': 'NULL', 'low_residual_volatility': 'NULL', 'momentum_12_1': 'NULL', 'post_earnings_drift': 'NULL', 'short_interest': 'NULL', 'short_term_reversal': 'NULL'}


## 1. Every criterion, its stored number and its verdict

In [3]:
# the criterion text is printed as stored, so a reworded threshold
# would show up here as a diff against sprints/E7/RESULTS.json
for name, block in stored["criteria"].items():
    print(name, block["verdict"])
    print(" ", block["criterion"])
    print(" ", json.dumps(block["stored_numbers"], sort_keys=True)[:220])

F7.1 pass
  Shift audit: moving every signal forward by one day flips or kills its IC. This proves the absence of leakage.
  {"idio_momentum": {"flipped": false, "killed": false, "leak_flag": false, "pit_by_construction": true, "t_lagged": 5.141445172653162, "t_next": 5.141445172653162, "t_now": 5.0435495844141}, "low_residual_volatility": {"f
F7.2 fail
  Factor-neutral momentum IC mean above 0.02 with t above 2, reported separately in-sample 2010 to 2020 and out-of-sample 2021 to 2026.
  {"in_sample": {"mean": -0.006059038694501872, "n": 108, "t": -0.7074147838845868}, "mean": -0.010683646861635072, "out_of_sample": {"mean": -0.01813823913104381, "n": 67, "t": -1.6387118619136074}, "t": -1.54900149736754
F7.3 pass
  Any signal failing the out-of-sample deflated-Sharpe hurdle is labeled NULL in the ledger, and the ledger contains at least as many rows as signal runs executed.
  {"below_hurdle_signals": {"low_residual_volatility": "NULL", "post_earnings_drift": "NULL", "short_interest":

## 2. By hand: the information coefficient on one day

In [4]:
# IC = rank-correlation(s_t, r_t), recomputed on one date from the
# stored signal and returns rather than copied from the IC artifact
from efb import alpha, eval_risk, hygiene, race

wide, _counts = eval_risk.load_clean_wide(DATA)
signal = alpha._signal_for("momentum_12_1", wide, DATA)
ic = hygiene.spearman_ic(signal, wide, 1)
date = ic.index[-1]
s_wide = signal.pivot(index="date", columns="ticker", values="signal")
s = s_wide.loc[date]
r = wide.loc[date]
both = pd.concat([s, r], axis=1).dropna()
by_hand = both.iloc[:, 0].rank().corr(both.iloc[:, 1].rank())
stored_ic = pd.read_parquet(
    ALPHA / "momentum_12_1" / "ic.parquet"
)["ic_h1"].loc[date]
print("IC by hand:", round(float(by_hand), 6))
print("IC stored:", round(float(stored_ic), 6))
assert abs(by_hand - float(stored_ic)) < 1e-9

IC by hand: -0.012216
IC stored: -0.012216


## 3. By hand: factor neutralization

In [5]:
# s_perp = s - X (X'X)^-1 X' s on one rebalance date; the residual must
# be orthogonal to every non-constant design column
grid = race.race_grid(DATA)
date = grid[-1]
neutral = hygiene.neutralize(signal, wide, date, DATA)
names = list(neutral["ticker"])
design = race._descriptor_design(date, names, DATA)
residual = neutral["signal"].to_numpy(dtype=float)
correlations = [
    abs(np.corrcoef(design[:, k], residual)[0, 1])
    for k in range(design.shape[1])
    if np.std(design[:, k]) > 0
]
print("max abs correlation with a design column:",
      float(max(correlations)))
assert max(correlations) < 1e-6

max abs correlation with a design column: 9.118698118466058e-16


## 4. The shift audit, run live

In [6]:
# the lagged-construction probe: every input moved one day back, the
# lagged IC against r_t decides leakage or fast decay
for name in ("momentum_12_1", "post_earnings_drift"):
    signal_i = alpha._signal_for(name, wide, DATA)
    lagged = alpha.lagged_signal(name, wide, DATA)
    audit = hygiene.shift_audit(signal_i, lagged, wide)
    print(name, "leak:", bool(audit["leak_flag"].iloc[-1]),
          "t_now:", round(float(audit["t_ic"].iloc[-1]), 2),
          "t_lagged:", round(float(audit["t_ic_lagged"].iloc[-1]), 2))

momentum_12_1 leak: False t_now: 4.46 t_lagged: 4.52


post_earnings_drift leak: True t_now: 14.05 t_lagged: 0.77


## 5. The D6 panel-to-column map, and the non-empty guard

In [7]:
from dashboard.tabs import d06_alpha_lab as d6  # noqa: E402

summary = d6.load_summary()
print("summary", summary.shape, list(summary.columns))
for name in d6.SIGNALS:
    path = d6.ALPHA / name / "ic.parquet"
    if not path.exists():
        continue
    assert not d6.ic_panel(name).empty
    print(name, d6.ic_panel(name).shape)
ledger = d6.ledger_panel()
assert not ledger.empty
print("ledger", ledger.shape)

summary (6, 26) ['signal', 'ic_h1_mean', 'ic_h1_t', 'ic_h5_mean', 'ic_h21_mean', 'ic_h63_mean', 'neutral_ic_h21_mean', 'neutral_ic_h21_t', 'in_sample_mean', 'in_sample_t', 'out_of_sample_mean', 'out_of_sample_t', 'oos_spread_sharpe', 'audit_flipped', 'audit_killed', 'audit_leak_flag', 'audit_mean_ic_next', 'audit_t_next', 'breadth', 'implied_ir', 'realized_ir', 'spread_sharpe', 'turnover_mean', 'hit_rate', 'audit_t_lagged', 'pit_by_construction']
momentum_12_1 (4193, 4)
short_term_reversal (4193, 4)
idio_momentum (3941, 4)
low_residual_volatility (3941, 4)
short_interest (4193, 4)
post_earnings_drift (4193, 4)
ledger (111, 11)


## 6. Evidence for the reports, in citation order

In [8]:
# the numbers the six reports cite, read from the artifacts, not typed
summary = pd.read_parquet(ALPHA / "summary.parquet")
print(summary[["signal", "ic_h1_mean", "ic_h1_t",
              "neutral_ic_h21_mean", "out_of_sample_t",
              "oos_spread_sharpe", "audit_leak_flag"]].to_string())
print(pd.read_parquet(
    ALPHA / "momentum_12_1" / "regime_ic.parquet"
).to_string())

                    signal  ic_h1_mean    ic_h1_t  neutral_ic_h21_mean  out_of_sample_t  oos_spread_sharpe  audit_leak_flag
0            idio_momentum    0.012102   5.043550            -0.003094         3.823912           0.395048            False
1  low_residual_volatility   -0.001400  -0.464478             0.001865        -0.638702          -0.870520            False
2            momentum_12_1    0.015076   4.461372            -0.010684         3.214804           0.270916            False
3      post_earnings_drift    0.123986  14.045657             0.009876         7.270038           0.035417             True
4           short_interest    0.002677   1.420239             0.008518         2.601971           0.855373            False
5      short_term_reversal    0.012031   4.708957             0.002796         2.440844          -0.484755            False
     regime   mean_ic  n_days
0    pooled  0.015076    3941
1   vix_low  0.014945    1317
2   vix_mid  0.018176    1310
3  vix_high 

## 7. What E8 inherits

In [9]:
# E8 sizes positions from the stored alpha contract; every signal is
# NULL, so construction runs on synthetic alpha with a known IC
contract = pd.read_parquet(ALPHA / "momentum_12_1" / "alpha.parquet")
print("alpha contract columns:", list(contract.columns))
print("gate verdicts:",
      {name: block["verdict"] for name, block in gate.items()})

alpha contract columns: ['date', 'ticker', 'alpha', 'alpha_xs_v2', 'ic', 'sigma_idio_xs_v1', 'sigma_idio_xs_v2', 'z', 'kappa']
gate verdicts: {'idio_momentum': 'NULL', 'low_residual_volatility': 'NULL', 'momentum_12_1': 'NULL', 'post_earnings_drift': 'NULL', 'short_interest': 'NULL', 'short_term_reversal': 'NULL'}


## 7b. Credit port note: what changes when the cross-section is bonds

In [10]:
# the IC and decay machinery is instrument-agnostic; the signals that
# exist only in equities (earnings drift) have no bond analogue, and
# the design columns are the credit factors instead of the equity ones
print("equity-only signals: post_earnings_drift, short_interest")
print("bond analogues: carry and roll-down instead of\n"
      "momentum and reversal")

equity-only signals: post_earnings_drift, short_interest
bond analogues: carry and roll-down instead of
momentum and reversal


## 8. Closing checklist

In [11]:
def numeric_leaves(node):
    out = []
    if isinstance(node, dict):
        for value in node.values():
            out.extend(numeric_leaves(value))
    elif isinstance(node, list):
        for value in node:
            out.extend(numeric_leaves(value))
    elif isinstance(node, float):
        out.append(node)
    return out


import nbformat

notebook = nbformat.read(
    ROOT / "notebooks" / "E7_walkthrough.ipynb", as_version=4
)
source = "\n".join(
    "".join(cell.source)
    for cell in notebook.cells
    if cell.cell_type == "code"
)
values = numeric_leaves(stored["criteria"]) + numeric_leaves(gate)
offenders = []
for value in values:
    for text in (f"{value:.6f}", f"{value:.4f}"):
        if len(text) > 6 and text in source:
            offenders.append(text)
print("stored values typed into a cell:", offenders)
assert offenders == []
assert all(
    name in source for name in ("F7.1", "F7.2", "F7.3", "F7.4")
)
print("closing checklist: clean")

stored values typed into a cell: []
closing checklist: clean
